# Lesson 03 - Agentic Design Patterns

ਇਸ ਪਾਠ ਵਿੱਚ, ਅਸੀਂ ਪ੍ਰਭਾਵਸ਼ਾਲੀ AI ਏਜੰਟ ਬਣਾਉਣ ਲਈ ਤਿੰਨ ਮੂਲ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ ਦੀ ਚਰਚਾ ਕਰਦੇ ਹਾਂ:

1. **ਸਾਫ਼ ਏਜੰਟ ਨਿਰਦੇਸ਼** — ਏਜੰਟ ਦੇ ਵਿਹਾਰ ਨੂੰ ਮਾਰਗਦਰਸ਼ਿਤ ਕਰਨ ਵਾਲੇ ਸਹੀ, ਭੂਮਿਕਾ-ਪਰਭਾਵਿਤ ਪ੍ਰੰਪਟ ਤਿਆਰ ਕਰਨਾ  
2. **Pydantic ਮਾਡਲਾਂ ਨਾਲ ਸਾਂਝੀਬੱਧ ਨਤੀਜੇ** — ਇਹ ਯਕੀਨੀ ਬਣਾਉਣਾ ਕਿ ਏਜੰਟ ਪੇਸ਼ ਕਰਨ ਵਾਲਾ ਡੇਟਾ ਭਵਿੱਖਵਾਣੀਯੋਗ ਅਤੇ ਮਾਨਤਾ ਪ੍ਰਾਪਤ ਹੈ  
3. **ਇੱਕਲ ਪ੍ਰਤਿਭਾਗ ਏਜੰਟ** — ਧਿਆਨ ਕੇਂਦਰਿਤ ਏਜੰਟ ਡਿਜ਼ਾਈਨ ਕਰਨਾ ਜੋ ਹਰ ਇੱਕ ਇੱਕ ਕੰਮ ਚੰਗੀ ਤਰ੍ਹਾਂ ਕਰਦੇ ਹਨ  

ਅਸੀਂ ਹਰ ਪੈਟਰਨ ਨੂੰ ਇੱਕ **ਯਾਤਰਾ ਮੰਜ਼ਿਲ ਸਿਫ਼ਾਰਸ਼ੀ ਪ੍ਰਣਾਲੀ** ਸੰਦਰਭ ਵਿੱਚ ਲਾਗੂ ਕਰਾਂਗੇ, ਜੋ ਕ੍ਰਮਵੱਧੀਕ ਤੌਰ 'ਤੇ ਇਕ ਸਿਸਟਮ ਬਣਾਉਂਦਾ ਹੈ ਜੋ ਮੰਜ਼ਿਲਾਂ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕਰ ਸਕਦਾ ਹੈ, ਉਪਲਬਧਤਾ ਜਾਂਚ ਸਕਦਾ ਹੈ ਅਤੇ ਲਾਜਿਸਟਿਕਸ ਨੂੰ ਸੰਭਾਲ ਸਕਦਾ ਹੈ।


## ਸੈਟਅਪ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## ਪੈਟਰਨ 1: ਸਾਫ-ਸੁਥਰੇ ਏਜੈਂਟ ਨਿਰਦੇਸ਼

ਸਭ ਤੋਂ ਪ੍ਰਭਾਵਸ਼ালী ਪੈਟਰਨ ਸਭ ਤੋਂ ਸਧਾਰਣ ਵੀ ਹੈ: ਆਪਣੇ ਏਜੈਂਟ ਲਈ ਸਾਫ਼, ਵਿਸਥਾਰਪੂਰਨ ਨਿਰਦੇਸ਼ ਲਿਖਣਾ।

ਚੰਗੇ ਨਿਰਦੇਸ਼ ਇਹ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੇ ਹਨ:
- **ਕੌਣ** ਏਜੈਂਟ ਹੈ (ਪਾਤਰ ਅਤੇ ਅੰਦਾਜ਼)
- **ਕੀ** ਕਰਨਾਂ ਚਾਹੀਦਾ ਹੈ (ਕਦਮ-ਦਰ-ਕਦਮ ਜਿੰਮੇਵਾਰੀਆਂ)
- **ਕਿਵੇਂ** ਵਤੀਰਾ ਕਰਨਾ ਚਾਹੀਦਾ ਹੈ (ਸੀਮਾਵਾਂ ਅਤੇ ਅੰਦਾਜ਼)

ਹੇਠਾਂ, ਅਸੀਂ ਇੱਕ ਯਾਤਰਾ ਸੰਭਾਲਣ ਵਾਲਾ ਏਜੈਂਟ ਬਣਾਉਂਦੇ ਹਾਂ ਜਿਸਦੇ ਸਪਸ਼ਟ ਨਿਰਦੇਸ਼ ਹਰ ਜਵਾਬ ਨੂੰ ਰੂਪ ਦਿੰਦੇ ਹਨ ਜੋ ਇਹ ਤਿਆਰ ਕਰਦਾ ਹੈ।


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Pydantic ਮਾਡਲਾਂ ਨਾਲ ਸੰਰਚਿਤ ਆਉਟਪੁੱਟ

ਮੁਕਤ ਰੂਪ ਦਾ ਪਾਠ ਗੱਲਬਾਤ ਲਈ ਲਾਭਦਾਇਕ ਹੈ, ਪਰ ਹੇਠਾਂ ਵਾਲੀਆਂ ਪ੍ਰਣਾਲੀਆਂ ਨੂੰ ਸੰਰਚਿਤ ਡੇਟਾ ਦੀ ਲੋੜ ਹੁੰਦੀ ਹੈ।  
**Pydantic ਮਾਡਲਾਂ** ਨੂੰ ਇੱਕ **ਟੂਲ ਫੰਕਸ਼ਨ** ਨਾਲ ਜੋੜ ਕੇ, ਅਸੀਂ:

- ਏਜੰਟ ਦੀ ਆਉਟਪੁੱਟ ਲਈ ਇੱਕ ਸਹੀ ਸਕੀਮਾ ਪਰਿਭਾਸ਼ਿਤ ਕਰ ਸਕਦੇ ਹਾਂ  
- ਜਵਾਬਾਂ ਨੂੰ ਆਪਣੇ ਆਪ ਸਹੀ ਹੋਣਾ ਚੈੱਕ ਕਰ ਸਕਦੇ ਹਾਂ  
- ਏਜੰਟ ਦੇ ਨਤੀਜੇ ਐਪਲੀਕੇਸ਼ਨ ਲੋਜਿਕ ਵਿੱਚ ਭਰੋਸੇਯੋਗ ਢੰਗ ਨਾਲ ਸ਼ਾਮਿਲ ਕਰ ਸਕਦੇ ਹਾਂ  

ਅਸੀਂ ਇੱਕ ਅਜਿਹਾ ਟੂਲ ਵੀ ਪੇਸ਼ ਕਰਦੇ ਹਾਂ ਜੋ ਮੰਜ਼ਿਲ ਦੇ ਵੇਰਵੇ ਵਾਪਸ ਕਰਦਾ ਹੈ ਤਾਂ ਜੋ ਏਜੰਟ ਆਪਣੇ ਸੁਝਾਵਾਂ ਨੂੰ ਅਸਲੀ ਡੇਟਾ 'ਤੇ ਅਧਾਰਤ ਕਰ ਸਕੇ।


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## ਪੈਟਰਨ 3: ਸਿੰਗਲ ਜ਼ਿੰਮੇਵਾਰੀ ਏਜੰਟ

ਜਟਿਲ ਕੰਮਾਂ ਨੂੰ ਕਈ ਕੇਂਦਰਿਤ ਏਜੰਟਾਂ ਵਿੱਚ ਵੰਡਣ ਨਾਲ ਫਾਇਦਾ ਹੁੰਦਾ ਹੈ, ਹਰ ਇੱਕ ਦੀ ਇੱਕ ਸਿੰਗਲ ਜ਼ਿੰਮੇਵਾਰੀ ਹੁੰਦੀ ਹੈ:

- ਇੱਕ **ਮੰਡੀਸ਼ Expert** ਜੋ ਸਥਾਨਾਂ ਅਤੇ ਉਪਲਬਧਤਾ ਬਾਰੇ ਜਾਣਦਾ ਹੈ
- ਇੱਕ **ਲੌਜਿਸਟਿਕਸ ਯੋਜਕ** ਜੋ ਉਡਾਣਾਂ, ਹੋਟਲਾਂ ਅਤੇ ਯਾਤਰੀ ਯੋਜਨਾਵਾਂ ਨੂੰ ਸੰਭਾਲਦਾ ਹੈ

ਇਹ ਸਾਫਟਵੇਅਰ ਇੰਜੀਨੀਅਰਿੰਗ ਦੇ ਸਿਧਾਂਤ *ਵਿਚਾਰਾਂ ਦੀ ਵੰਡ* ਨੂੰ ਦਰਸਾਉਂਦਾ ਹੈ — ਹਰ ਏਜੰਟ ਅਲੱਗ-ਅਲੱਗ ਤੌਰ 'ਤੇ ਟੈਸਟ ਕਰਨ, ਸੰਭਾਲਣ ਅਤੇ ਸੁਧਾਰਨ ਲਈ ਅਸਾਨ ਹੁੰਦਾ ਹੈ।


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## ਸਰੰਸ਼

ਇਸ ਪਾਠ ਵਿੱਚ ਅਸੀਂ ਇੱਕ ਯਾਤਰਾ ਸਿਫਾਰਸ਼ਕ ਸਥਿਤੀ ਲਈ ਤਿੰਨ ਏਜੈਂਟਿਕ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ ਲਾਗੂ ਕੀਤੇ:

| ਪੈਟਰਨ | ਮੁੱਖ ਵਿਚਾਰ | ਫਾਇਦਾ |
|---|---|---|
| **ਸਾਫ਼ ਹੁਕਮ** | ਪਹਿਲਾਂ ਹੀ ਪੈਰੋਨਾ, ਜਿੰਮੇਵਾਰੀਆਂ ਅਤੇ ਸੀਮਾਵਾਂ ਨਿਰਧਾਰਤ ਕਰੋ | ਲਗਾਤਾਰ, ਬ੍ਰਾਂਡ-ਅਨੁਕੂਲ ਏਜੈਂਟ ਵਿਹਾਰ |
| **ਸੰਰਚਿਤ ਨਤੀਜਾ** | ਜਵਾਬ ਹੋਣ ਦੇ ਰੂਪ ਵਿੱਚ ਪਾਇਡੈਂਟਿਕ ਮਾਡਲਾਂ ਦੀ ਵਰਤੋਂ ਕਰੋ | ਪ੍ਰਮਾਣਿਤ, ਮਸ਼ੀਨ-ਪੜ੍ਹਨ ਯੋਗ ਨਤੀਜੇ |
| **ਇੱਕਲوج਼ਮੀ ਜ਼ਿੰਮੇਵਾਰੀ** | ਹਰ ਏਜੈਂਟ ਨੂੰ ਇੱਕ ਕੇਂਦਰਿਤ ਕੰਮ ਦਿਓ | ਆਸਾਨੀ ਨਾਲ ਟੈਸਟ ਕਰਨ, ਦੇਖਭਾਲ ਕਰਨ ਅਤੇ ਮਿਲਾ ਕੇ ਬਣਾਉਣ ਯੋਗ |

ਇਹ ਪੈਟਰਨ ਕੁਦਰਤੀ ਤੌਰ 'ਤੇ ਮਿਲ ਕੇ ਕੰਮ ਕਰਦੇ ਹਨ — ਤੁਸੀਂ ਸਾਫ਼ ਹੁਕਮਾਂ ਨੂੰ ਸੰਰਚਿਤ ਨਤੀਜੇ ਨਾਲ ਇਕਲوج਼ਮੀ ਜ਼ਿੰਮੇਵਾਰੀ ਵਾਲੇ ਏਜੈਂਟ ਦੇ ਅੰਦਰ ਜੋੜ ਕੇ ਮਜ਼ਬੂਤ, ਉਤਪਾਦਨ-ਤਿਆਰ ਪ੍ਰਣਾਲੀਆਂ ਬਣਾ ਸਕਦੇ ਹੋ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪੱਤਰ**:
ਇਹ ਦਸਤਾਵੇਜ਼ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਅਨੁਵਾਦਿਤ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਅਤਾ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਵਿੱਚ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਥਿਰਤਾਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਪ੍ਰਮਾਣਿਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜ਼ਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਨਾਲ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਅਸੀਂ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
